In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [ ]:
llm=ChatOpenAI()

In [8]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [9]:
def generate_joke(state: JokeState):
    prompt=f"Generate a joke on the topic {state["topic"]}"
    response=llm.invoke(prompt).content

    return {"joke": response}

In [10]:
def generate_explanation(state: JokeState):
    prompt=f"Write a explanation for the joke {state["joke"]}"
    response=llm.invoke(prompt).content

    return {"joke": response}

In [12]:
mygraph=StateGraph(JokeState)

mygraph.add_node("generate_joke", generate_joke)
mygraph.add_node("generate_explanation", generate_explanation)

mygraph.add_edge(START, "generate_joke")
mygraph.add_edge("generate_joke", "generate_explanation")
mygraph.add_edge("generate_explanation", END)

checkpointer=InMemorySaver()

workflow=mygraph.compile(checkpointer=checkpointer)

In [ ]:
config1={"configurable": {"thread_id":"1"}}
workflow.invoke({"topic": "pizza"}, config=config1)

In [ ]:
# Now we can get state of this thread1
workflow.getstate(cofig1)

In [ ]:
# We can also see intermediate state value at each checkpoint

list(workflow.get_state_history(config1))

In [ ]:
# Now we can also create new thread to maintain seperate memory
config2={"configurable": {"thread_id":"2"}}
worflow.invoke({"topic": "pasta"}, config=config2)